In [1]:
cd /

/


In [2]:
%ls

'=0.3.7'                      kaggle_requirements.txt    python-apt.tar.xz*
 bin@                         lib@                       requirements.txt
 boot/                        lib32@                     root/
 colab_requirements.txt       lib64@                     run/
 content/                     libx32@                    sbin@
 cuda-keyring_1.0-1_all.deb   media/                     srv/
 datalab/                     mnt/                       sys/
 dev/                         NGC-DL-CONTAINER-LICENSE   tmp/
 etc/                         opt/                       tools/
 home/                        proc/                      usr/
 kaggle/                      python-apt/                var/


In [3]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.7/914.7 kB 25.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
cp -r /kaggle/input/ultralytics/ultralytics ultralytics

In [5]:
import cv2
import os
from PIL import Image
from ultralytics import YOLO
from shapely.geometry import Polygon, box
import matplotlib.pyplot as plt
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
model = YOLO("/kaggle/input/model1/best.pt")
# accepts all formats - image/dir/Path/URL/video/PIL/ndarray. 0 for webcam
results = model.predict(source="/kaggle/input/predict-picture/pic-828_jpg.rf.4aa182469c26efc3b21ec696fd0ad540.jpg",imgsz=320,save = True)
# results = model.predict(source="folder", show=True)  # Display preds. Accepts all YOLO predict arguments

/usr/local/lib/python3.10/dist-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


image 1/1 /kaggle/input/predict-picture/pic-828_jpg.rf.4aa182469c26efc3b21ec696fd0ad540.jpg: 320x320 5 curbs, 719.5ms
Speed: 11.7ms preprocess, 719.5ms inference, 22.6ms postprocess per image at shape (1, 3, 320, 320)
Results saved to runs/segment/predict


In [7]:
results_dir = "runs/segment/predict4"

# 获取预测后的图片文件名
predicted_image_file = os.listdir(results_dir)[0]  # 假设文件夹中只有一张图片

# 构建完整路径
predicted_image_path = os.path.join(results_dir, predicted_image_file)

# 使用 PIL 加载图片
image = Image.open(predicted_image_path)

# 使用 Matplotlib 显示图片
plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.axis('off')
plt.title("YOLO Prediction Result")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'runs/segment/predict4'

In [ ]:
masks = results[0].masks
mask_data = []
# 检查是否有掩码
if masks is not None:
    # 遍历所有掩码并获取归一化坐标
    for i, mask in enumerate(masks.xyn):
        #print(f"Mask {i + 1} normalized coordinates:")
        mask_data = np.vstack(mask)
        print(mask_data)
else:
    print("No masks found in the prediction results.")

In [ ]:
def create_grid():
    """Create offset chessboard grid"""
    rows = [5, 6, 5, 6, 5]  # 每行的格子数量
    grid = []
    y_start = 0  # 从(0,0)开始
    height = -1 / len(rows)  # 计算每个格子的高度
    
    for row_idx, cols in enumerate(rows):
        width = 1 / cols  # 计算每个格子的宽度
        x_start = (1 - cols * width) / 2  # 使矩阵左右对齐
        for col_idx in range(cols):
            x1, y1 = x_start + col_idx * width, y_start
            x2, y2 = x1 + width, y1 + height
            grid.append((x1, y1, x2, y2))
        y_start += height
    
    return grid

def normalize_coordinates(mask_coords):
    """Normalization"""
    return [(float(x), -float(y)) for x, y in mask_coords]  # 确保数据转换为 float

def extract_masks(masks):
    """Extract data"""
    extracted_masks = []
    if masks is not None:
        for mask in masks.xyn:  # 遍历所有 masks
            extracted_masks.append(normalize_coordinates(mask.tolist()))  # 转换并存储
    return extracted_masks

def find_intersecting_cells(grid, masks):
    """Find intersection"""
    grid_polygons = [Polygon([(x1, y1), (x2, y1), (x2, y2), (x1, y2)]) for (x1, y1, x2, y2) in grid]
    intersected_cells = set()
    
    for mask_coords in masks:
        mask_polygon = Polygon(mask_coords)
        for i, cell in enumerate(grid_polygons):
            if mask_polygon.intersects(cell):
                intersected_cells.add(i + 1)  # 格子编号从1开始
    
    return sorted(intersected_cells)

def visualize(grid, masks, intersected_cells):
    fig, ax = plt.subplots(figsize=(6, 6))
    
    for i, (x1, y1, x2, y2) in enumerate(grid):
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, edgecolor='black', facecolor='none', linewidth=1)
        ax.add_patch(rect)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(i+1), ha='center', va='center', fontsize=8)
    
    for mask in masks:
        mask_polygon = Polygon(mask)
        x, y = mask_polygon.exterior.xy
        ax.fill(x, y, color='blue', alpha=0.3, label='Mask' if 'Mask' not in ax.get_legend_handles_labels()[1] else "")
    
    for idx in intersected_cells:
        x1, y1, x2, y2 = grid[idx-1]
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, edgecolor='red', facecolor='red', alpha=0.5)
        ax.add_patch(rect)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(-1, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Mask & Grid Intersection")
    ax.legend()
    plt.show()

#run
grid = create_grid()
masks = results[0].masks
mask_data = extract_masks(masks)
intersected_cells = find_intersecting_cells(grid, mask_data)
print("Intersected Grid Cells:", intersected_cells)

# visualize
visualize(grid, mask_data, intersected_cells)